In [1]:
import findspark
findspark.init()

# 1. 用户行为创建ALS模型

## 1.1 创建Spark会话

In [2]:
# spark配置信息
from pyspark import SparkConf
from pyspark.sql import SparkSession

SPARK_APP_NAME = "RecommendationSystem"

conf = SparkConf()    # 创建spark config对象
config = (
    ("spark.app.name", SPARK_APP_NAME),    # 设置启动的spark的app名称，没有提供，将随机产生一个名称
    ("spark.executor.memory", "8g"),    # 设置该app启动时占用的内存用量，默认1g
    ("spark.executor.cores", "4"),    # 设置spark executor使用的CPU核心数
    # 以下三项配置，可以控制执行器数量
#     ("spark.dynamicAllocation.enabled", True),
#     ("spark.dynamicAllocation.initialExecutors", 1),    # 1个执行器
#     ("spark.shuffle.service.enabled", True)
# 	('spark.sql.pivotMaxValues', '99999'),  # 当需要pivot DF，且值很多时，需要修改，默认是10000
)
# 查看更详细配置及说明：https://spark.apache.org/docs/latest/configuration.html

conf.setAll(config)

# 利用config对象，创建spark session
spark = SparkSession.builder.config(conf=conf).getOrCreate()

## 1.2 从hdfs中加载csv文件为DataFrame

In [5]:
# 从hdfs加载CSV文件为DataFrame
df = spark.read.csv("hdfs://localhost:9000/E_commerce_platform/action_2.csv", header=True)
df.show()    # 查看dataframe，默认显示前20条
# 大致查看一下数据类型
df.printSchema()    # 打印当前dataframe的结构

+-------+--------+------+----------+
|item_id| user_id|action|     vtime|
+-------+--------+------+----------+
|      1| 1144920| click|2013-09-18|
|      1|13739929| click|2013-09-26|
|      1| 9989957| click|2013-09-26|
|      1| 4722735| click|2013-09-14|
|      1| 5379171| click|2013-09-18|
|      1|10541267| click|2013-09-30|
|      1| 2224274| click|2013-09-30|
|      1|11564774| click|2013-09-17|
|      1| 2224274| click|2013-09-30|
|      1| 5372775| click|2013-09-07|
|      1|11419007| click|2013-09-25|
|      1| 8032677| click|2013-09-29|
|      1| 8600885| click|2013-09-13|
|      1| 6833292| click|2013-09-29|
|      1| 6833292| click|2013-09-29|
|      1|14476749| click|2013-09-27|
|      1| 3190209| click|2013-09-29|
|      1| 2612833| click|2013-09-30|
|      1| 5379171| click|2013-09-26|
|      1| 5822327| click|2013-09-18|
+-------+--------+------+----------+
only showing top 20 rows

root
 |-- item_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-

## 1.3 设置结构

In [8]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, FloatType
from pyspark.sql.functions import unix_timestamp
from pyspark.sql.functions import from_unixtime
# 构建结构对象
schema = StructType([
    StructField("item_id", IntegerType()),
    StructField("user_id", IntegerType()),
    StructField("action", StringType()),
    StructField("vtime", StringType())
])
# 从hdfs加载数据为dataframe，并设置结构
action_df = spark.read.csv("hdfs://localhost:9000/E_commerce_platform/action_2.csv", header=True, schema=schema)
action_df.show()
# review_df.count() 
action_df.printSchema() 

+-------+--------+------+----------+
|item_id| user_id|action|     vtime|
+-------+--------+------+----------+
|      1| 1144920| click|2013-09-18|
|      1|13739929| click|2013-09-26|
|      1| 9989957| click|2013-09-26|
|      1| 4722735| click|2013-09-14|
|      1| 5379171| click|2013-09-18|
|      1|10541267| click|2013-09-30|
|      1| 2224274| click|2013-09-30|
|      1|11564774| click|2013-09-17|
|      1| 2224274| click|2013-09-30|
|      1| 5372775| click|2013-09-07|
|      1|11419007| click|2013-09-25|
|      1| 8032677| click|2013-09-29|
|      1| 8600885| click|2013-09-13|
|      1| 6833292| click|2013-09-29|
|      1| 6833292| click|2013-09-29|
|      1|14476749| click|2013-09-27|
|      1| 3190209| click|2013-09-29|
|      1| 2612833| click|2013-09-30|
|      1| 5379171| click|2013-09-26|
|      1| 5822327| click|2013-09-18|
+-------+--------+------+----------+
only showing top 20 rows

root
 |-- item_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 

## 1.4 分析数据集字段

In [5]:
print("查看item_id的数据情况：", action_df.groupBy("item_id").count().count())
# 1百万个商品
#review_df.groupBy("item_id").count()  返回的是一个dataframe，这里的count计算的是每一个分组的个数，但当前还没有进行计算
# 当调用df.count()时才开始进行计算，这里的count计算的是dataframe的条目数，也就是共有多少个分组

查看item_id的数据情况： 1000000


In [6]:
print("查看user_id的数据情况：", action_df.groupBy("user_id").count().count())
#9380925个用户

查看user_id的数据情况： 9380925


In [7]:
print("查看action的数据情况：", action_df.groupBy("action").count().collect())    # collect会把计算结果全部加载到内存，谨慎使用
# 只有4种类型数据

查看action的数据情况： [Row(action='alipay', count=1548054), Row(action='cart', count=4939123), Row(action='click', count=151759491), Row(action='collect', count=3970091)]


In [10]:
# 统计每个用户对每一种商品的alipay、cart、click、collect数量
cate_count_df = action_df.groupBy(action_df.item_id, action_df.user_id).pivot("action",["alipay","cart","click","collect"]).count()
cate_count_df.printSchema()    # 此时还没有开始计算

root
 |-- item_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- alipay: long (nullable = true)
 |-- cart: long (nullable = true)
 |-- click: long (nullable = true)
 |-- collect: long (nullable = true)



In [11]:
# 由于运算时间比较长，所以这里先将结果存储起来，供后续其他操作使用
# 写入数据时才开始计算
cate_count_df.write.csv("hdfs://localhost:9000/E_commerce_platform/action/cate_count.csv", header=True)

In [9]:
spark.stop()

## 1.5 根据用户对每件商品偏好打分训练ALS模型

In [12]:
from pyspark import SparkConf
from pyspark.sql import SparkSession
# 创建Spark会话
SPARK_APP_NAME = "ALS_Model_Training"

conf = SparkConf()    # 创建spark config对象
config = (
    ("spark.app.name", SPARK_APP_NAME),    # 设置启动的spark的app名称，没有提供，将随机产生一个名称
    ("spark.executor.memory", "8g"),    # 设置该app启动时占用的内存用量，默认1g
    ("spark.executor.cores", "4"),    # 设置spark executor使用的CPU核心数
)
conf.setAll(config)

# 利用config对象，创建spark session
spark = SparkSession.builder.config(conf=conf).getOrCreate()

In [13]:
spark.sparkContext.setCheckpointDir("hdfs://localhost:9000/checkPoint/")
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, FloatType

# 构建结构对象
schema = StructType([
    StructField("user_id", IntegerType()),
    StructField("item_id", IntegerType()),
    StructField("alipay", IntegerType()),
    StructField("cart", IntegerType()),
    StructField("click", IntegerType()),
    StructField("collect", IntegerType())
])

# 从hdfs加载CSV文件
cate_count_df = spark.read.csv("hdfs://localhost:9000/E_commerce_platform/action/cate_count.csv", header=True, schema=schema)
cate_count_df.printSchema()
cate_count_df.first()    # 第一行数据

root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- alipay: integer (nullable = true)
 |-- cart: integer (nullable = true)
 |-- click: integer (nullable = true)
 |-- collect: integer (nullable = true)



Row(user_id=61, item_id=685220, alipay=None, cart=None, click=4, collect=1)

In [14]:
def process_row(r):
    # 处理每一行数据：r表示row对象
    # 偏好评分规则：
    # 如果行为次数小于等于20次，则计算得分为行为次数乘以对应的权重（0.2、0.4、0.6、1.0）
    # 如果行为次数大于20次，则得分固定为4、8、12、20
    
    # 注意这里要全部设为浮点数，spark运算时对类型比较敏感，要保持数据类型都一致
    click_count = r.click if r.click else 0.0
    collect_count = r.collect if r.collect else 0.0
    cart_count = r.cart if r.cart else 0.0
    alipay_count = r.alipay if r.alipay else 0.0

    click_score = 0.2*click_count if click_count<=20 else 4.0
    collect_score = 0.4*collect_count if collect_count<=20 else 8.0
    cart_score = 0.6*cart_count if cart_count<=20 else 12.0
    alipay_count = 1.0*alipay_count if alipay_count<=20 else 20.0

    rating = click_score + collect_score + cart_score + alipay_count
    # 返回用户ID、商品ID、用户对分类的偏好打分
    return r.user_id, r.item_id, rating

In [ ]:
返回一个PythonRDD类型

In [15]:
# 返回一个PythonRDD类型，此时还没开始计算
cate_count_df.rdd.map(process_row).toDF(["user_id", "item_id", "rating"])

DataFrame[user_id: bigint, item_id: bigint, rating: double]

In [ ]:
用户对商品的打分数据

In [16]:
# 用户对商品的打分数据
# map返回的结果是rdd类型，需要调用toDF方法转换为Dataframe
cate_rating_df = cate_count_df.rdd.map(process_row).toDF(["user_id", "item_id", "rating"])
# 注意：toDF不是每个rdd都有的方法，仅局限于此处的rdd
# 可通过该方法获得 user-cate-matrix
# 但由于cateId字段过多，这里运算量比很大，机器内存要求很高才能执行，否则无法完成任务
# 请谨慎使用

# 但好在我们训练ALS模型时，不需要转换为user-cate-matrix，所以这里可以不用运行
# cate_rating_df.groupBy("userId").povit("cateId").min("rating")
# 用户对类别的偏好打分数据
cate_rating_df

DataFrame[user_id: bigint, item_id: bigint, rating: double]

In [17]:
# 分批次训练ALS模型
num_batches = 10
batch_size = int(cate_rating_df.count() / num_batches)
print(batch_size)

11307266


In [18]:
from pyspark.ml.recommendation import ALS


# 初始化ALS模型
als = ALS(userCol='user_id', itemCol='item_id', ratingCol='rating', checkpointInterval=5, maxIter=8)
# 创建空列表保存所有模型
models = []

# 分批次训练ALS模型
for i in range(num_batches):
    batch_data = cate_rating_df.limit(batch_size).cache()
    model = als.fit(batch_data)
    models.append(model)
    batch_data.unpersist()
    
    print(f"Batch {i+1} training completed")

# 合并所有模型为最终模型
final_model = models[0]
for m in models[1:]:
    final_model = final_model.union(m)

Py4JJavaError: An error occurred while calling o260.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 41.0 failed 1 times, most recent failure: Lost task 0.0 in stage 41.0 (TID 582, LAPTOP-L36Q0L8U, executor driver): java.lang.OutOfMemoryError: Java heap space
	at java.io.ObjectInputStream$HandleTable.grow(ObjectInputStream.java:4035)
	at java.io.ObjectInputStream$HandleTable.assign(ObjectInputStream.java:3842)
	at java.io.ObjectInputStream.readArray(ObjectInputStream.java:2127)
	at java.io.ObjectInputStream.readObject0(ObjectInputStream.java:1698)
	at java.io.ObjectInputStream.readArray(ObjectInputStream.java:2160)
	at java.io.ObjectInputStream.readObject0(ObjectInputStream.java:1698)
	at java.io.ObjectInputStream.defaultReadFields(ObjectInputStream.java:2472)
	at java.io.ObjectInputStream.readSerialData(ObjectInputStream.java:2396)
	at java.io.ObjectInputStream.readOrdinaryObject(ObjectInputStream.java:2254)
	at java.io.ObjectInputStr

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 13896)
Traceback (most recent call last):
  File "F:\Anaconda\procedure\lib\socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "F:\Anaconda\procedure\lib\socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "F:\Anaconda\procedure\lib\socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "F:\Anaconda\procedure\lib\socketserver.py", line 747, in __init__
    self.handle()
  File "D:\home\spark\spark-3.0.0-bin-hadoop2.7\spark-3.0.0-bin-hadoop2.7\python\pyspark\accumulators.py", line 268, in handle
    poll(accum_updates)
  File "D:\home\spark\spark-3.0.0-bin-hadoop2.7\spark-3.0.0-bin-hadoop2.7\python\pyspark\accumulators.py", line 241, in poll
    if func():
  File "D:\home\spark\spark-3.0.0-bin-hadoop2.7\spark-3.0.

In [ ]:
# model.recommendForAllUsers(N) 给所有用户推荐TOP-N个物品
ret = final_model.recommendForAllUsers(3)
# 由于是给所有用户进行推荐，此处运算时间也较长
ret.show()
# 推荐结果存放在recommendations列中，
ret.select("recommendations").show()

In [ ]:
评估

In [ ]:
使用 RegressionEvaluator 类来评估模型的均方根误差 (RMSE)。

In [ ]:
# 拆分数据集为训练集和测试集
train_data, test_data = data.randomSplit([0.8, 0.2])

# 显示训练集和测试集大小
print("Train data size:", train_data.count())
print("Test data size:", test_data.count())

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

# 进行预测
predictions = model.transform(test_data)

# 创建评估器
evaluator = RegressionEvaluator(labelCol="score", predictionCol="prediction", metricName="rmse")

# 计算 RMSE
rmse = evaluator.evaluate(predictions)
print("RMSE:", rmse)
